# RETIA-DSS: train và test trên Google Colab

Notebook này clone code từ GitHub vào ổ đĩa nhanh của Colab, nhưng lưu checkpoint bền vững trong Google Drive. Dữ liệu đã có sẵn trong repo nên không cần upload riêng.

Trước khi chạy: chọn **Runtime > Change runtime type**, đặt **Runtime Version = 2026.07** và **Hardware accelerator = T4 GPU** (hoặc GPU mạnh hơn). RETIA chưa có DGL wheel cho runtime Python 3.13 mới nhất.

In [ ]:
import sys
print('Python:', sys.version)
if sys.version_info[:2] != (3, 12):
    raise RuntimeError(
        'Notebook cần Python 3.12. Vào Runtime > Change runtime type > '
        'Runtime Version = 2026.07, chọn GPU, rồi chạy lại từ đầu.'
    )
!nvidia-smi


## 1. Cài môi trường tương thích

Mã RETIA gốc dùng PyTorch/DGL rất cũ. Cell trước phải xác nhận Python 3.12 thành công trước khi chạy cell này. Ta dùng PyTorch 2.4.1 + CUDA 12.4 và DGL 2.4.0; không import `torch` trước khi cài xong.

In [ ]:
%pip uninstall -y torch torchvision torchaudio torchdata dgl
%pip install --no-cache-dir torch==2.4.1 --index-url https://download.pytorch.org/whl/cu124
%pip install --no-cache-dir "dgl==2.4.0+cu124" -f https://data.dgl.ai/wheels/torch-2.4/cu124/repo.html
%pip install --no-cache-dir tqdm scipy pandas rdflib matplotlib


In [ ]:
import sys, torch, dgl
print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('DGL:', dgl.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Chưa có GPU. Vào Runtime > Change runtime type > GPU rồi chạy lại.')
print('GPU:', torch.cuda.get_device_name(0))


## 2. Mount Drive, clone repo và nối thư mục checkpoint

Code và data chạy tại `/content/RETIA-DSS`; checkpoint trong `MyDrive/RETIA-DSS/models`. Nếu runtime mất kết nối, checkpoint vẫn còn trên Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import os, shutil, subprocess

REPO_URL = 'https://github.com/DangHung1110/RETIA-DSS.git'
WORKSPACE = Path('/content/RETIA-DSS')
DRIVE_ROOT = Path('/content/drive/MyDrive/RETIA-DSS')
CHECKPOINT_DIR = DRIVE_ROOT / 'models'
LOG_DIR = DRIVE_ROOT / 'logs'
PLOT_DIR = DRIVE_ROOT / 'plots'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)

if not (WORKSPACE / '.git').exists():
    subprocess.run(['git', 'clone', REPO_URL, str(WORKSPACE)], check=True)
else:
    subprocess.run(['git', '-C', str(WORKSPACE), 'pull', '--ff-only'], check=True)

models_path = WORKSPACE / 'models'
if models_path.is_symlink():
    models_path.unlink()
elif models_path.exists():
    shutil.copytree(models_path, CHECKPOINT_DIR, dirs_exist_ok=True)
    shutil.rmtree(models_path)
models_path.symlink_to(CHECKPOINT_DIR, target_is_directory=True)

print('Repo:', WORKSPACE)
print('Checkpoint:', models_path.resolve())
print('Logs:', LOG_DIR)
print('Plots:', PLOT_DIR)
print('Data folders:', sorted(p.name for p in (WORKSPACE / 'data').iterdir() if p.is_dir()))


## 3. Kiểm tra nhanh trước khi train thật

Cell này chỉ chạy 2 epoch với mô hình nhỏ (`n_layers=1`, `n_hidden=32`) để bắt lỗi môi trường. Checkpoint của nó có tên khác mô hình chính. Có thể mất vài phút.

In [ ]:
smoke_cmd = [
    sys.executable, 'main.py', '-d', 'YAGO',
    '--train-history-len', '3', '--test-history-len', '3',
    '--lr', '0.001', '--n-layers', '1', '--n-hidden', '32',
    '--n-epochs', '2', '--evaluate-every', '1',
    '--self-loop', '--decoder', 'convtranse', '--encoder', 'uvrgcn',
    '--layer-norm', '--gpu', '0', '--ft_lr=0.001',
    '--norm_weight', '1', '--task-weight', '0.7'
]
subprocess.run(smoke_cmd, cwd=WORKSPACE / 'src', check=True)


## 4. Train mô hình chính

Bắt đầu với `YAGO` vì lệnh đơn giản, không cần static graph. Sau đó có thể đổi `DATASET` thành `WIKI`, `ICEWS14`, `ICEWS18` hoặc `ICEWS05-15`. Giữ nguyên toàn bộ tham số khi test để tên checkpoint khớp.

In [ ]:
DATASET = 'YAGO'  # @param ['YAGO', 'WIKI', 'ICEWS14', 'ICEWS18', 'ICEWS05-15']
N_EPOCHS = 250

DATASET_CONFIG = {
    'YAGO':       {'history': 3, 'extra': []},
    'WIKI':       {'history': 3, 'extra': []},
    'ICEWS14':    {'history': 9, 'extra': ['--weight', '0.5', '--angle', '14', '--discount', '1', '--add-static-graph']},
    'ICEWS18':    {'history': 4, 'extra': ['--weight', '0.5', '--angle', '10', '--discount', '1', '--add-static-graph']},
    'ICEWS05-15': {'history': 9, 'extra': ['--weight', '0.5', '--angle', '10', '--discount', '1', '--add-static-graph']},
}
cfg = DATASET_CONFIG[DATASET]

base_cmd = [
    sys.executable, 'main.py', '-d', DATASET,
    '--train-history-len', str(cfg['history']),
    '--test-history-len', str(cfg['history']),
    '--lr', '0.001', '--n-layers', '2', '--evaluate-every', '1',
    '--n-hidden', '200', '--self-loop', '--decoder', 'convtranse',
    '--encoder', 'uvrgcn', '--layer-norm', '--gpu', '0',
    '--ft_lr=0.001', '--norm_weight', '1', '--task-weight', '0.7',
] + cfg['extra']

def run_and_log(command, log_path, cwd):
    log_path = Path(log_path)
    print('Command:', ' '.join(command))
    print('Log:', log_path)
    with log_path.open('w', encoding='utf-8', buffering=1) as log_file:
        process = subprocess.Popen(
            command, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1,
        )
        for line in process.stdout:
            print(line, end='')
            log_file.write(line)
        return_code = process.wait()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, command)
    return log_path

print('Dataset:', DATASET, '| history:', cfg['history'], '| epochs:', N_EPOCHS)


Chỉ chạy cell dưới khi muốn train hoặc train lại. Nếu Drive đã có checkpoint và chỉ muốn test/vẽ metric, hãy bỏ qua cell này.

In [ ]:
train_cmd = base_cmd + ['--n-epochs', str(N_EPOCHS)]
TRAIN_LOG = run_and_log(
    train_cmd, LOG_DIR / f'{DATASET}_train.log', WORKSPACE / 'src'
)


## 5. Offline test

Chạy sau khi train xong. Kết quả cần ghi lại là `MRR` và `Hits@1/3/10`; ưu tiên dòng `filter_ent` khi báo cáo link prediction thực thể.

In [ ]:
test_cmd = base_cmd + ['--test']
TEST_LOG = run_and_log(
    test_cmd, LOG_DIR / f'{DATASET}_offline_test.log', WORKSPACE / 'src'
)


## 6. Biểu đồ huấn luyện và đánh giá

Với link prediction, `accuracy` thông thường không phải chỉ số chính. Các chỉ số chuẩn là MRR và Hits@1/3/10. Cell dưới đọc log, vẽ loss, Best MRR theo epoch, MRR/Hits cuối cùng, rồi lưu CSV và PNG vào Drive. Biểu đồ loss chỉ có nếu lần train được chạy bằng notebook có ghi log này.

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
train_log_path = LOG_DIR / f'{DATASET}_train.log'
test_log_path = LOG_DIR / f'{DATASET}_offline_test.log'

epoch_pattern = re.compile(
    r'Epoch\s+(\d+)\s+\|\s+Ave Loss:([-+0-9.eE]+)\s+\|\s+'
    r'entity-relation-static:([-+0-9.eE]+)-([-+0-9.eE]+)-([-+0-9.eE]+)'
    r'\s+Best MRR\s+([-+0-9.eE]+)'
)

if train_log_path.exists():
    train_text = train_log_path.read_text(encoding='utf-8', errors='replace')
    train_rows = [
        {
            'epoch': int(m.group(1)), 'total_loss': float(m.group(2)),
            'entity_loss': float(m.group(3)), 'relation_loss': float(m.group(4)),
            'static_loss': float(m.group(5)), 'best_mrr': float(m.group(6)),
        }
        for m in epoch_pattern.finditer(train_text)
    ]
    train_df = pd.DataFrame(train_rows)
    if len(train_df):
        train_df.to_csv(LOG_DIR / f'{DATASET}_training_history.csv', index=False)
        fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
        axes[0].plot(train_df.epoch, train_df.total_loss, label='Total loss', linewidth=2)
        axes[0].plot(train_df.epoch, train_df.entity_loss, label='Entity loss', alpha=0.85)
        axes[0].plot(train_df.epoch, train_df.relation_loss, label='Relation loss', alpha=0.85)
        if train_df.static_loss.abs().max() > 0:
            axes[0].plot(train_df.epoch, train_df.static_loss, label='Static loss', alpha=0.85)
        axes[0].set(title=f'{DATASET} training losses', xlabel='Epoch', ylabel='Loss')
        axes[0].legend()
        axes[1].plot(train_df.epoch, train_df.best_mrr, color='tab:green', linewidth=2)
        axes[1].set(title='Best validation MRR', xlabel='Epoch', ylabel='MRR', ylim=(0, 1))
        fig.tight_layout()
        fig.savefig(PLOT_DIR / f'{DATASET}_training_curves.png', dpi=180, bbox_inches='tight')
        plt.show()
    else:
        print('Không tìm thấy dòng epoch trong train log.')
else:
    print('Chưa có train log. Checkpoint cũ không chứa loss history; cần train lại nếu muốn vẽ loss.')

if test_log_path.exists():
    test_text = test_log_path.read_text(encoding='utf-8', errors='replace')
    values = {}
    for method, value in re.findall(r'MRR \(([^)]+)\):\s*([-+0-9.eE]+)', test_text):
        values.setdefault(method, {})['MRR'] = float(value)
    for method, hit, value in re.findall(r'Hits \(([^)]+)\) @ (\d+):\s*([-+0-9.eE]+)', test_text):
        values.setdefault(method, {})[f'Hits@{hit}'] = float(value)
    metric_df = pd.DataFrame.from_dict(values, orient='index')
    metric_df.index.name = 'evaluation'
    metric_df.to_csv(LOG_DIR / f'{DATASET}_offline_metrics.csv')
    display(metric_df.round(6))

    fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
    mrr_order = [m for m in ['raw_ent', 'filter_ent', 'raw_rel', 'filter_rel'] if m in metric_df.index]
    axes[0].bar(mrr_order, metric_df.loc[mrr_order, 'MRR'], color=['tab:blue', 'tab:cyan', 'tab:orange', 'tab:red'][:len(mrr_order)])
    axes[0].set(title=f'{DATASET} offline MRR', xlabel='Evaluation protocol', ylabel='MRR', ylim=(0, 1))
    axes[0].tick_params(axis='x', rotation=20)

    hit_cols = [c for c in ['Hits@1', 'Hits@3', 'Hits@10'] if c in metric_df.columns]
    hit_methods = [m for m in ['filter_ent', 'filter_rel'] if m in metric_df.index]
    x = np.arange(len(hit_cols))
    width = 0.36
    for idx, method in enumerate(hit_methods):
        offset = (idx - (len(hit_methods) - 1) / 2) * width
        axes[1].bar(x + offset, metric_df.loc[method, hit_cols], width, label=method)
    axes[1].set(title='Filtered Hits@K', xlabel='K', ylabel='Hits', ylim=(0, 1), xticks=x, xticklabels=hit_cols)
    axes[1].legend()
    fig.tight_layout()
    fig.savefig(PLOT_DIR / f'{DATASET}_offline_metrics.png', dpi=180, bbox_inches='tight')
    plt.show()
else:
    print('Chưa có offline test log. Hãy chạy mục 5 trước; không cần train lại.')


## 7. Online continual training/test (tùy chọn)

Thứ tự đúng là: train tổng quát → `--test-valid` → `--test-test`. Bước online tạo nhiều checkpoint theo từng timestamp và tốn thêm dung lượng Drive.

In [ ]:
valid_cmd = base_cmd + ['--test-valid']
VALID_ONLINE_LOG = run_and_log(
    valid_cmd, LOG_DIR / f'{DATASET}_online_valid.log', WORKSPACE / 'src'
)

online_test_cmd = base_cmd + ['--test-test']
ONLINE_TEST_LOG = run_and_log(
    online_test_cmd, LOG_DIR / f'{DATASET}_online_test.log', WORKSPACE / 'src'
)


In [ ]:
import re
import pandas as pd
import matplotlib.pyplot as plt

online_log_path = LOG_DIR / f'{DATASET}_online_test.log'
if online_log_path.exists():
    online_text = online_log_path.read_text(encoding='utf-8', errors='replace')
    online_mrr = [
        float(v) for v in re.findall(
            r'Continual Model\s*:\s*\*\*\*test mrr\*\*\*\s*([-+0-9.eE]+)',
            online_text,
        )
    ]
    with open(WORKSPACE / 'data' / DATASET / 'test.txt', encoding='utf-8') as handle:
        test_timestamps = sorted({int(line.split('\t')[3]) for line in handle if line.strip()})
    if len(test_timestamps) != len(online_mrr):
        test_timestamps = list(range(len(online_mrr)))
    online_df = pd.DataFrame({'timestamp': test_timestamps, 'filtered_entity_mrr': online_mrr})
    online_df.to_csv(LOG_DIR / f'{DATASET}_online_mrr_by_timestamp.csv', index=False)
    display(online_df.round(6))
    ax = online_df.plot(
        x='timestamp', y='filtered_entity_mrr', marker='o', linewidth=2,
        figsize=(9, 4.8), legend=False,
        title=f'{DATASET} continual filtered entity MRR',
    )
    ax.set(xlabel='Test timestamp', ylabel='Filtered entity MRR', ylim=(0, 1))
    fig = ax.get_figure()
    fig.tight_layout()
    fig.savefig(PLOT_DIR / f'{DATASET}_online_mrr_by_timestamp.png', dpi=180, bbox_inches='tight')
    plt.show()
else:
    print('Chưa có online test log. Hãy chạy cell --test-test ngay phía trên.')


In [ ]:
print('Các checkpoint đã lưu:')
for path in sorted(CHECKPOINT_DIR.rglob('*')):
    if path.is_file() and path.suffix != '.txt':
        print(path.relative_to(CHECKPOINT_DIR), f'{path.stat().st_size / 1024**2:.1f} MB')


## 8. Demo dự đoán Top-K trên YAGO

Demo lấy một fact thật trong test set, chỉ đưa `(subject, relation, ?)` cho checkpoint general và hiển thị 10 object có điểm cao nhất. Đổi `TIME_INDEX` từ 0–5 và `QUERY_INDEX` để xem ví dụ khác. Cell này không dùng object đúng để tạo điểm dự đoán; object chỉ được dùng sau đó để tính rank.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

TIME_INDEX = 0   # 0..5, tương ứng timestamp YAGO 183..188
QUERY_INDEX = 0  # fact thứ mấy trong timestamp đó
TOP_K = 10

os.chdir(WORKSPACE / 'src')
if str(WORKSPACE) not in sys.path:
    sys.path.insert(0, str(WORKSPACE))

from rgcn import utils
from rgcn.utils import build_sub_graph, get_relhead_reltal, build_super_g
from src.rrgcn import RecurrentRGCN

def read_id_names(path):
    result = {}
    with open(path, encoding='utf-8') as handle:
        for line in handle:
            parts = line.rstrip('\n').split('\t')
            result[int(parts[1])] = parts[0].strip('<>')
    return result

data = utils.load_data('YAGO')
train_list = utils.split_by_time(data.train)
valid_list = utils.split_by_time(data.valid)
test_list = utils.split_by_time(data.test)
test_times = sorted(np.unique(data.test[:, 3]).tolist())
if not 0 <= TIME_INDEX < len(test_list):
    raise ValueError(f'TIME_INDEX phải nằm trong 0..{len(test_list)-1}')
if not 0 <= QUERY_INDEX < len(test_list[TIME_INDEX]):
    raise ValueError(f'QUERY_INDEX phải nằm trong 0..{len(test_list[TIME_INDEX])-1}')

entity_name = read_id_names(WORKSPACE / 'data/YAGO/entity2id.txt')
relation_name = read_id_names(WORKSPACE / 'data/YAGO/relation2id.txt')
device = torch.device('cuda:0')
num_nodes, num_rels = data.num_nodes, data.num_rels

model = RecurrentRGCN(
    'convtranse', 'uvrgcn', num_nodes, num_rels, 200, 'sub',
    sequence_len=3, num_bases=100, num_basis=100,
    num_hidden_layers=2, dropout=0.2, self_loop=True,
    skip_connect=False, layer_norm=True, input_dropout=0.2,
    hidden_dropout=0.2, feat_dropout=0.2, aggregation='none',
    use_cuda=True, gpu=0, analysis=False, segnn=False, dataset='YAGO',
    kg_layer=2, bn=False, comp_op='mul', ent_drop=0.2, rel_drop=0.1,
    num_words=0, num_static_rels=0, weight=1, discount=1, angle=10,
    use_static=False,
).to(device)
checkpoint_path = CHECKPOINT_DIR / 'YAGO' / 'YAGO-uvrgcn-convtranse-ly2-his3-dp0.2|0.2|0.2|0.2-gpu0'
checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=True)
model.load_state_dict(checkpoint['state_dict'])
model.eval()

history = train_list + valid_list
history_len = 3
if TIME_INDEX - history_len < 0:
    input_list = history[TIME_INDEX - history_len:] + test_list[:TIME_INDEX]
else:
    input_list = test_list[TIME_INDEX - history_len:TIME_INDEX]

history_graphs = [build_sub_graph(num_nodes, num_rels, snap, True, False, 0) for snap in input_list]
history_super_graphs = []
for snap in input_list:
    rel_head, rel_tail = get_relhead_reltal(snap, num_nodes, num_rels)
    history_super_graphs.append(build_super_g(num_rels, rel_head, rel_tail, True, False, 0))

subject_id, relation_id, answer_id = map(int, test_list[TIME_INDEX][QUERY_INDEX])
query = torch.tensor([[subject_id, relation_id, 0]], dtype=torch.long, device=device)
_, entity_scores, _ = model.predict(
    history_graphs, history_super_graphs, num_rels, None, query, True
)
tail_scores = entity_scores[0]
top_scores, top_ids = torch.topk(tail_scores, k=TOP_K)
answer_rank = int((tail_scores > tail_scores[answer_id]).sum().item()) + 1

print(f'Timestamp: {test_times[TIME_INDEX]}')
print(f'Query: ({entity_name[subject_id]}, {relation_name[relation_id]}, ?)')
print(f'Ground truth: {entity_name[answer_id]} | rank = {answer_rank}/{num_nodes}')
top_ids_cpu = top_ids.detach().cpu().tolist()
top_scores_cpu = top_scores.detach().cpu().tolist()
demo_df = pd.DataFrame({
    'rank': range(1, TOP_K + 1),
    'entity_id': top_ids_cpu,
    'predicted_object': [entity_name[int(i)] for i in top_ids_cpu],
    'score': top_scores_cpu,
    'is_ground_truth': [int(i) == answer_id for i in top_ids_cpu],
})
display(demo_df)

fig, ax = plt.subplots(figsize=(10, max(4.8, TOP_K * 0.52)))
plot_df = demo_df.iloc[::-1]
colors = ['tab:green' if flag else 'tab:blue' for flag in plot_df.is_ground_truth]
bars = ax.barh(plot_df.predicted_object, plot_df.score, color=colors)
ax.set(
    title=f'Top-{TOP_K}: {entity_name[subject_id]} — {relation_name[relation_id]} — ?',
    xlabel='Aggregated RETIA score', ylabel='Candidate object',
)
for bar, value in zip(bars, plot_df.score):
    ax.text(value, bar.get_y() + bar.get_height()/2, f' {value:.4f}', va='center', fontsize=9)
fig.tight_layout()
demo_plot_path = PLOT_DIR / f'YAGO_demo_t{test_times[TIME_INDEX]}_q{QUERY_INDEX}.png'
fig.savefig(demo_plot_path, dpi=180, bbox_inches='tight')
plt.show()
print('Saved:', demo_plot_path)


## Lỗi thường gặp

- `CUDA available: False`: chưa chọn GPU runtime.
- `No matching distribution for dgl`: kiểm tra Python là 3.12 hoặc 3.11 và chạy lại cell cài đặt.
- Test lại chạy train: checkpoint không tồn tại hoặc tham số train/test không khớp nên tên file khác.
- Colab ngắt giữa chừng: chạy lại từ đầu; repo được clone lại nhưng checkpoint vẫn ở Drive. Code hiện tại chưa hỗ trợ resume giữa epoch, nên sẽ train lại general training từ epoch 0.
- Hết dung lượng Drive ở bước online: xóa các checkpoint theo timestamp không còn cần thiết, giữ lại checkpoint general tốt nhất.